# AMEX Enterprise Credit Risk Platform
## Notebook 41 -- Dynamic / Behavioral Credit Scoring: Financial-Impact Reporting & Packaging
### Phase 3 . Problem Statement 6: Dynamic / Behavioral Credit Scoring

CRISP-DM stage: **Evaluation & Deployment (reporting)**. Sprint 1, Notebook 4 of 4 for this problem -- the final notebook of Problem 6. Depends on Problem 1 Notebooks 05/08 (real champion model name, and the real, inherited EAD/LGD assumptions) and this problem's own Notebooks 38-40 (policy, modeling results, validation & deployment outcome). Reuses the exact reporting/packaging template proven in Notebook 29 (Problem 4), Notebook 33 (Problem 3), and Notebook 37 (Problem 5).

**What this notebook does (real, computed on your machine when you run it):**
- States Problem 6's own financial narrative -- **Real Recency-Detection Value**: unlike Problem 5's Notebook 37 (which had to DERIVE an estimated holdout defaulter count from the platform's overall default rate), this notebook uses Notebook 40's REAL, EXACT confusion matrix at the F1-optimal threshold -- every true-positive and false-positive count below is measured, not estimated
- Inherits EAD and LGD from Problem 1's Notebook 08 rather than re-guessing them
- Computes real loss-prevention opportunity **net of an explicit false-positive review cost** -- a refinement over Problem 5's report, made possible because Notebook 40's real confusion matrix makes the false-positive count directly measurable, not just the true-positive count
- Computes Year-1 ROI and payback period from an explicit, fully editable set of financial assumptions (a monthly re-scoring cadence, matching Notebook 38's original "dynamic/monthly refreshed" business framing -- different from Problem 5's quarterly early-window cadence, since these are different use cases: new-account screening vs. ongoing existing-book monitoring)
- Writes SMART suggestions for six organizational levels, from frontline behavioral-monitoring analysts up to the CFO, each referencing real numbers from this problem's own notebooks (including Notebook 40's bootstrap CIs, calibration gap, and PSI)
- Produces a Word report, a formula-linked Excel workbook (editing an assumption recalculates every downstream dollar figure -- independently verified by recalculating the workbook and confirming every formula matches the notebook's own printed values exactly), and an interactive HTML dashboard
- When available in this environment, includes an honest recent-vs-early comparison chart against Problem 5's real results at the same window length (informational only -- Problem 6 does not depend on Problem 5)
- Handles the honest edge case plainly: if the estimated annual NET benefit is ever $0 or negative (a real possibility here, since false-positive review cost is now netted against loss-prevention benefit), ROI and payback are reported as "N/A -- no measurable net benefit under current assumptions" everywhere, rather than crashing or fabricating a number

**What this notebook does NOT do:** train or validate any model -- that's Notebooks 39 and 40. This is reporting and packaging only.

Zero-fabrication: every number in this report is computed live from this run's real upstream notebook outputs; only the assumptions explicitly labeled ASSUMPTION are illustrative and meant to be edited to your institution's real figures.

**This is the final notebook of Problem 6 -- once it completes, Problem 6 (Dynamic/Behavioral Credit Scoring) is complete. Problems 7 (Early Warning System) and 8 (Roll-Rate Modeling) remain to close out Phase 3.**

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD NOTEBOOKS 05/08/38/39/40's REAL OUTPUTS
# =============================================================================
import json
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Notebooks 05/08/38/39/40's Real Outputs")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
P6_ROOT = PROJECT_ROOT / "Phase3_Behavioral_Intelligence" / "Problem6_Dynamic_Behavioral_Credit_Scoring"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

PILLAR_DIRS = {
    "p6_policy": P6_ROOT / "policy",
    "p6_modeling": P6_ROOT / "modeling",
    "p6_deployment": P6_ROOT / "deployment",
    "p6_reporting_packaging": P6_ROOT / "financial_impact_reporting_packaging",
}
for _d in PILLAR_DIRS.values():
    _d.mkdir(parents=True, exist_ok=True)

P1_CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB05_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_05_summary.json"
NB08_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_08_summary.json"
NB38_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_38_summary.json"
NB39_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_39_summary.json"
NB40_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_40_summary.json"

for _p, _fix in [
    (P1_CONFIG_PATH, "run Problem 1's Notebook 01 first."),
    (NB05_SUMMARY_PATH, "run Problem 1's Notebook 05 first."),
    (NB08_SUMMARY_PATH, "run Problem 1's Notebook 08 first (this notebook inherits its real "
                         "EAD/LGD assumptions rather than re-guessing them)."),
    (NB38_SUMMARY_PATH, "run 38_dynamic_behavioral_scoring_business_understanding.ipynb first."),
    (NB39_SUMMARY_PATH, "run 39_dynamic_behavioral_scoring_modeling.ipynb first."),
    (NB40_SUMMARY_PATH, "run 40_dynamic_behavioral_scoring_validation_deployment.ipynb first."),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p.name} not found in its expected location.\nFix: {_fix}")

with open(P1_CONFIG_PATH, "r", encoding="utf-8") as f:
    P1_CONFIG = json.load(f)
with open(NB05_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB05_SUMMARY = json.load(f)
with open(NB08_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB08_SUMMARY = json.load(f)
with open(NB38_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB38_SUMMARY = json.load(f)
with open(NB39_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB39_SUMMARY = json.load(f)
with open(NB40_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB40_SUMMARY = json.load(f)

POLICY_PATH = Path(NB38_SUMMARY["policy_path"])
with open(POLICY_PATH, "r", encoding="utf-8") as f:
    DBS_POLICY = json.load(f)

MODELING_RESULTS_PATH = Path(NB39_SUMMARY["modeling_results_path"])
with open(MODELING_RESULTS_PATH, "r", encoding="utf-8") as f:
    MODELING_ARTIFACT = json.load(f)

WINNING_W = NB40_SUMMARY["winning_w"]
MEETS_KPI = NB40_SUMMARY["meets_kpi_target"]
RECOMMENDED_FOR_PRODUCTION = NB40_SUMMARY["recommended_for_production"]
WINNING_W_RESULT = MODELING_ARTIFACT["results_by_w"][str(WINNING_W)]

CHAMPION_NAME = NB05_SUMMARY["champion_model"]
FULL_HISTORY_AUC = MODELING_ARTIFACT["full_history_reference_auc"]
EAD_PER_ACCOUNT_USD = NB08_SUMMARY["ead_per_account_usd_assumption"]
LGD_ASSUMPTION = NB08_SUMMARY["lgd_assumption"]

# --- These threshold-dependent metrics (Notebook 40's F1-optimal-threshold
#     reproduction) give this problem something Problem 5's own Notebook 37
#     did not have: a REAL, EXACT confusion matrix on the real holdout
#     population -- not a derived/estimated defaulter count. This report
#     uses that real confusion matrix directly rather than re-deriving an
#     estimate from the platform's overall default rate. ---
METRICS_AT_F1_OPTIMAL = NB40_SUMMARY["metrics_at_f1_optimal_threshold"]
METRICS_AT_050 = NB40_SUMMARY["metrics_at_threshold_0_50"]

print(f"Winning trailing window (Notebook 40)     : W={WINNING_W}")
print(f"Meets KPI target / recommended for prod   : {MEETS_KPI} / {RECOMMENDED_FOR_PRODUCTION}")
print(f"Reproduced holdout AUC (Notebook 40)       : {NB40_SUMMARY['reproduced_holdout_auc']:.4f} "
      f"(full history: {FULL_HISTORY_AUC:.4f})")
print(f"Real holdout population (F1-optimal confusion matrix, Notebook 40): "
      f"{sum(METRICS_AT_F1_OPTIMAL['confusion_matrix'].values()):,}")
print(f"EAD per account (Notebook 08, inherited)  : ${EAD_PER_ACCOUNT_USD:,}")
print(f"LGD assumption (Notebook 08, inherited)   : {LGD_ASSUMPTION:.0%}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: LIBRARY IMPORTS
# =============================================================================
_section("SECTION 2: Library Imports")

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

missing = []
try:
    import pandas as pd
except ImportError:
    missing.append("pandas")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    missing.append("matplotlib")
try:
    from docx import Document
    from docx.shared import Inches
    from docx.enum.text import WD_ALIGN_PARAGRAPH
except ImportError:
    missing.append("python-docx")
try:
    import openpyxl
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.worksheet.table import Table, TableStyleInfo
    from openpyxl.chart import BarChart, Reference
except ImportError:
    missing.append("openpyxl")
if missing:
    raise ImportError("Missing required package(s): " + ", ".join(missing) +
                       "\nFix: pip install " + " ".join(missing))

print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: FINANCIAL PLANNING ASSUMPTIONS (EXPLICIT, EDITABLE)
# =============================================================================
_section("SECTION 3: Financial Planning Assumptions (Explicit, Editable)")

# --- This dataset has no real collections-intervention outcomes or project-cost data.
#     Every ASSUMPTION-labeled figure below is stated and editable -- nothing here is
#     fabricated as if it were measured. EAD/LGD are real inherited values (read
#     programmatically from Problem 1's Notebook 08), not re-guessed here. ---
FINANCIAL_ASSUMPTIONS = {
    "ead_per_account_usd": {"value": EAD_PER_ACCOUNT_USD, "source": "Notebook 08 (inherited, real value read programmatically)"},
    "lgd_assumption": {"value": LGD_ASSUMPTION, "source": "Notebook 08 (inherited, real value read programmatically)"},
    "behavioral_intervention_success_rate": {
        "value": 0.20,
        "source": "ASSUMPTION -- illustrative efficacy of a recency-triggered intervention (credit-line "
                   "reduction, proactive outreach, reserve pre-build) on an EXISTING account flagged by "
                   "recent behavioral deterioration; edit to your institution's own outcome data.",
    },
    "false_positive_review_cost_usd": {
        "value": 35,
        "source": "ASSUMPTION -- illustrative staff-time cost of manually reviewing one account the model "
                   "flags that does NOT go on to default (a false positive at the F1-optimal threshold) -- "
                   "unlike Problem 5's report, this one nets this cost against the loss-prevention benefit "
                   "since Notebook 40's real confusion matrix makes the false-positive count directly "
                   "measurable, not just the true-positive count.",
    },
    "implementation_cost_usd": {
        "value": 60_000,
        "source": "ASSUMPTION -- illustrative one-time build/validate/deploy cost for the dynamic/behavioral "
                   "scoring service (data science + risk review time); edit to your institution's actual "
                   "project cost.",
    },
    "annual_application_cycles": {
        "value": 12,
        "source": "ASSUMPTION -- monthly re-scoring cadence, matching Notebook 38's original 'monthly "
                   "refreshed PD' business framing for this problem (Problem 5's early-window model, by "
                   "contrast, used a quarterly cadence -- these are different use cases: new-account "
                   "screening vs. ongoing existing-book monitoring); edit to your institution's actual cadence.",
    },
}
INTERVENTION_SUCCESS_RATE = FINANCIAL_ASSUMPTIONS["behavioral_intervention_success_rate"]["value"]
FALSE_POSITIVE_REVIEW_COST_USD = FINANCIAL_ASSUMPTIONS["false_positive_review_cost_usd"]["value"]
IMPLEMENTATION_COST_USD = FINANCIAL_ASSUMPTIONS["implementation_cost_usd"]["value"]
ANNUAL_APPLICATION_CYCLES = FINANCIAL_ASSUMPTIONS["annual_application_cycles"]["value"]

assumptions_path = PILLAR_DIRS["p6_reporting_packaging"] / "financial_assumptions.json"
with open(assumptions_path, "w", encoding="utf-8") as f:
    json.dump(FINANCIAL_ASSUMPTIONS, f, indent=2)
for _k, _v in FINANCIAL_ASSUMPTIONS.items():
    print(f"  {_k}: {_v['value']}  ({_v['source']})")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: REAL RECENCY-DETECTION VALUE -- POPULATION FLAGGED (EXACT, FROM
#            NOTEBOOK 40'S REPRODUCED CONFUSION MATRIX -- NOT DERIVED)
# =============================================================================
_section("SECTION 4: Real Recency-Detection Value -- Population Flagged")

# --- Unlike Problem 5's Notebook 37 (which had to DERIVE an estimated
#     holdout defaulter count from the platform's overall default rate,
#     since Notebooks 35/36 didn't persist per-customer labels), Notebook 40
#     reproduced the model and computed a REAL confusion matrix on the real
#     holdout set -- so every count below is exact, not estimated. The
#     F1-optimal threshold is used as the primary operating point (same
#     honesty caveat Notebook 39/40 already state: chosen on this same
#     holdout, reported for interpretability). The standard 0.5-threshold
#     matrix is also shown for comparison. ---
_cm_f1 = METRICS_AT_F1_OPTIMAL["confusion_matrix"]
_cm_050 = METRICS_AT_050["confusion_matrix"]
N_HOLDOUT = sum(_cm_f1.values())
N_HOLDOUT_DEFAULTERS = _cm_f1["tp"] + _cm_f1["fn"]  # real, exact: all true positives in this holdout

TRUE_POSITIVES_FLAGGED = _cm_f1["tp"]
FALSE_POSITIVES_FLAGGED = _cm_f1["fp"]
FLAGGED_TOTAL = TRUE_POSITIVES_FLAGGED + FALSE_POSITIVES_FLAGGED
RECENCY_CAPTURE_RATE = TRUE_POSITIVES_FLAGGED / N_HOLDOUT_DEFAULTERS if N_HOLDOUT_DEFAULTERS else 0.0

print(f"F1-optimal operating threshold (Notebook 40, real)   : {METRICS_AT_F1_OPTIMAL['threshold']:.4f}")
print(f"Real holdout population                              : {N_HOLDOUT:,}")
print(f"Real holdout defaulters (tp + fn, exact)              : {N_HOLDOUT_DEFAULTERS:,}")
print(f"Real true positives flagged (tp, exact)                : {TRUE_POSITIVES_FLAGGED:,}")
print(f"Real false positives flagged (fp, exact)               : {FALSE_POSITIVES_FLAGGED:,}")
print(f"Total accounts flagged for review at this threshold    : {FLAGGED_TOTAL:,}")
print(f"Real defaulter capture rate at this threshold          : {RECENCY_CAPTURE_RATE:.1%}")
print(f"\nFor comparison, at the standard 0.5 threshold: tp={_cm_050['tp']:,}, fp={_cm_050['fp']:,}, "
      f"capture rate={_cm_050['tp'] / N_HOLDOUT_DEFAULTERS:.1%}" if N_HOLDOUT_DEFAULTERS else "")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: LOSS-PREVENTION OPPORTUNITY, NET OF FALSE-POSITIVE REVIEW COST
# =============================================================================
_section("SECTION 5: Loss-Prevention Opportunity, Net of False-Positive Review Cost")

PREVENTABLE_DEFAULTS = round(TRUE_POSITIVES_FLAGGED * INTERVENTION_SUCCESS_RATE)
GROSS_LOSS_PREVENTED_USD = PREVENTABLE_DEFAULTS * EAD_PER_ACCOUNT_USD * LGD_ASSUMPTION
FALSE_POSITIVE_COST_USD = FALSE_POSITIVES_FLAGGED * FALSE_POSITIVE_REVIEW_COST_USD
NET_BENEFIT_PER_CYCLE_USD = GROSS_LOSS_PREVENTED_USD - FALSE_POSITIVE_COST_USD

print(f"True positives flagged (real, exact)                   : {TRUE_POSITIVES_FLAGGED:,}")
print(f"ASSUMPTION behavioral-intervention success rate         : {INTERVENTION_SUCCESS_RATE:.0%}")
print(f"Estimated preventable defaults                          : {PREVENTABLE_DEFAULTS:,}")
print(f"Gross loss prevented (this holdout sample, per cycle)   : ${GROSS_LOSS_PREVENTED_USD:,.0f}")
print(f"False positives flagged (real, exact)                   : {FALSE_POSITIVES_FLAGGED:,}")
print(f"ASSUMPTION cost per false-positive review                : ${FALSE_POSITIVE_REVIEW_COST_USD:,}")
print(f"Total false-positive review cost (per cycle)             : ${FALSE_POSITIVE_COST_USD:,.0f}")
print(f"Net benefit per cycle (gross loss prevented - FP cost)   : ${NET_BENEFIT_PER_CYCLE_USD:,.0f}")
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: ROI, INVESTMENT & PAYBACK PERIOD
# =============================================================================
_section("SECTION 6: ROI, Investment & Payback Period")

ANNUAL_BENEFIT_USD = NET_BENEFIT_PER_CYCLE_USD * ANNUAL_APPLICATION_CYCLES
ROI_PCT = ((ANNUAL_BENEFIT_USD - IMPLEMENTATION_COST_USD) / IMPLEMENTATION_COST_USD) * 100 if IMPLEMENTATION_COST_USD else None
PAYBACK_MONTHS = (IMPLEMENTATION_COST_USD / (ANNUAL_BENEFIT_USD / 12)) if ANNUAL_BENEFIT_USD > 0 else None
# Honest fallback text/values for the case where the estimated annual NET
# benefit is zero or negative (a real possibility here, unlike Problem 5's
# report, since this one nets the false-positive review cost against the
# loss-prevention benefit) -- rather than crashing on a None format or a
# nonsensical negative payback period, every downstream consumer (narrative
# text, Word report, Excel, JSON summary, HTML dashboard) uses these safe
# strings/values.
ROI_DISPLAY = f"{ROI_PCT:,.0f}%" if ROI_PCT is not None else "N/A"
PAYBACK_DISPLAY = (f"{PAYBACK_MONTHS:.1f} months" if PAYBACK_MONTHS
                    else "N/A (no measurable net benefit under current assumptions)")
PAYBACK_MONTHS_JSON = round(PAYBACK_MONTHS, 2) if PAYBACK_MONTHS else None
ROI_PCT_JSON = round(ROI_PCT, 1) if ROI_PCT is not None else None

print(f"Amount invested (ASSUMPTION, one-time)          : ${IMPLEMENTATION_COST_USD:,.0f}")
print(f"Estimated annual net benefit                     : ${ANNUAL_BENEFIT_USD:,.0f} "
      f"(= net benefit/cycle x {ANNUAL_APPLICATION_CYCLES} cycles/year, ASSUMPTION cadence)")
print(f"Estimated ROI (Year 1)                           : {ROI_DISPLAY}")
print(f"Estimated payback period                         : {PAYBACK_DISPLAY}")
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: SMART SUGGESTIONS -- BOTTOM TO TOP MANAGEMENT
# =============================================================================
_section("SECTION 7: SMART Suggestions -- Bottom to Top Management")

SMART_SUGGESTIONS = [
    {"org_level": "Behavioral Monitoring Ops / Frontline Risk Analysts",
     "suggestion": f"Work the {FLAGGED_TOTAL:,}-account flag list from the W={WINNING_W} recency model "
                   f"({RECENCY_CAPTURE_RATE:.1%} real defaulter capture at the F1-optimal threshold) each "
                   f"monthly re-scoring cycle -- this is an EXISTING-book monitoring signal, not a new-"
                   f"account screen, so route flags to account review, not new-application underwriting."},
    {"org_level": "Portfolio Risk Team Lead",
     "suggestion": f"Track the {PREVENTABLE_DEFAULTS:,}-account behavioral-intervention goal (from the "
                   f"{INTERVENTION_SUCCESS_RATE:.0%} ASSUMPTION success rate) and the "
                   f"{FALSE_POSITIVES_FLAGGED:,} real false positives (review-cost exposure) as paired "
                   f"weekly KPIs -- both numbers came from the same real confusion matrix, so tightening "
                   f"the threshold trades one against the other."},
    {"org_level": "Risk / Credit Analyst",
     "suggestion": f"Re-run this trailing-window scoring monthly as new statement data lands; monitor the "
                   f"split-half score PSI (Notebook 40 measured {NB40_SUMMARY['split_half_score_psi']:.4f}) "
                   f"and the bootstrap AUC CI (last measured "
                   f"[{NB40_SUMMARY['bootstrap_auc_ci'][0]:.4f}, {NB40_SUMMARY['bootstrap_auc_ci'][1]:.4f}]) "
                   f"-- re-validate if either drifts materially."},
    {"org_level": "Model Risk / Compliance (SR 11-7)",
     "suggestion": f"File Notebook 40's bootstrap AUC/PR-AUC CIs, calibration gap "
                   f"({NB40_SUMMARY['mean_calibration_gap']:.4f}), and AUC-retention KPI result "
                   f"({'MET' if MEETS_KPI else 'NOT MET'}) with the model's annual validation packet; this "
                   f"model is currently {'RECOMMENDED' if RECOMMENDED_FOR_PRODUCTION else 'NOT RECOMMENDED'} "
                   f"for production use, and is explicitly a RECENCY score complementary to Problem 1's "
                   f"full-history champion, not a replacement for it."},
    {"org_level": "Finance / Provisioning Team",
     "suggestion": f"Use the {TRUE_POSITIVES_FLAGGED:,} recency-flagged real defaulters to trigger early "
                   f"reserve-timing reviews on accounts whose FULL-HISTORY score may not yet reflect their "
                   f"recent deterioration -- coordinate with Problem 3's ECL work and Problem 4's "
                   f"tier-differentiated LGD for the $ reserve amount per flagged account."},
    {"org_level": "CFO / Executive Leadership",
     "suggestion": f"Approve the ${IMPLEMENTATION_COST_USD:,.0f} investment given an estimated "
                   f"{PAYBACK_DISPLAY} payback and {ROI_DISPLAY} Year-1 ROI (net of estimated false-"
                   f"positive review cost) from monthly behavioral re-scoring; revisit the "
                   f"{ANNUAL_APPLICATION_CYCLES}x/year cadence ASSUMPTION at the next quarterly business "
                   f"review."},
]
smart_df = pd.DataFrame(SMART_SUGGESTIONS)
smart_path = PILLAR_DIRS["p6_reporting_packaging"] / "p6_smart_suggestions.csv"
smart_df.to_csv(smart_path, index=False)
for _row in SMART_SUGGESTIONS:
    print(f"[{_row['org_level']}]\n  {_row['suggestion']}\n")
print(f"✅ Saved -> {smart_path.name}")
print("\n✅ Section 7 complete.")


# =============================================================================
# SECTION 8: INLINE CHARTS
# =============================================================================
_section("SECTION 8: Inline Charts")

VIZ = {"ink": "#0B1F3A", "accent": "#C41E3A", "muted": "#8A93A6", "gold": "#C9A227", "surface": "#FFFFFF"}

fig1, ax1 = plt.subplots(figsize=(7, 5), dpi=150)
_labels1 = ["True Positives\n(real defaulters flagged)", "False Positives\n(review cost)"]
_vals1 = [TRUE_POSITIVES_FLAGGED, FALSE_POSITIVES_FLAGGED]
_bars = ax1.bar(_labels1, _vals1, color=[VIZ["accent"], VIZ["muted"]])
for _b, _v in zip(_bars, _vals1):
    ax1.text(_b.get_x() + _b.get_width() / 2, _v, f"{_v:,}", ha="center", va="bottom", fontsize=11)
ax1.set_ylabel("Real holdout customers (exact confusion-matrix counts)")
ax1.set_title(f"Problem 6: Population Flagged at the F1-Optimal Threshold (W={WINNING_W})")
fig1.tight_layout()
chart1_path = PILLAR_DIRS["p6_reporting_packaging"] / "population_flagged_chart.png"
fig1.savefig(chart1_path, dpi=150, facecolor=VIZ["surface"])
plt.close(fig1)

# --- Second chart: honest recent-vs-early comparison, when Problem 5's real
#     results were available in this environment for Notebook 39 to compare
#     against (see Notebook 38 Section 6's secondary requirement). Skipped
#     gracefully (with a note) if not present -- Problem 6 does not depend
#     on Problem 5, so this is informational only. ---
_recent_vs_early = MODELING_ARTIFACT.get("recent_vs_early_comparison")
if _recent_vs_early:
    _ws_sorted = sorted(int(w) for w in _recent_vs_early.keys())
    _recent_aucs = [_recent_vs_early[str(w)]["recent_auc"] for w in _ws_sorted]
    _early_aucs = [_recent_vs_early[str(w)]["early_auc"] for w in _ws_sorted]
    fig2, ax2 = plt.subplots(figsize=(7, 5), dpi=150)
    _x = range(len(_ws_sorted))
    _bar_w = 0.35
    ax2.bar([i - _bar_w / 2 for i in _x], _recent_aucs, _bar_w, label="Recent (Problem 6)", color=VIZ["accent"])
    ax2.bar([i + _bar_w / 2 for i in _x], _early_aucs, _bar_w, label="Early (Problem 5)", color=VIZ["gold"])
    ax2.set_xticks(list(_x))
    ax2.set_xticklabels([f"W=K={w}" for w in _ws_sorted])
    ax2.set_ylabel("Holdout AUC")
    ax2.set_title("Problem 6 vs. Problem 5: Recent vs. Early Behavioral Signal\n(real, same window length)", fontsize=11)
    ax2.legend(fontsize=9)
    fig2.tight_layout()
    chart2_path = PILLAR_DIRS["p6_reporting_packaging"] / "recent_vs_early_comparison_chart.png"
    fig2.savefig(chart2_path, dpi=150, facecolor=VIZ["surface"])
    plt.close(fig2)
    print(f"✅ Saved -> {chart1_path.name}, {chart2_path.name}")
else:
    chart2_path = None
    print(f"✅ Saved -> {chart1_path.name}")
    print("(Recent-vs-early comparison chart skipped -- Problem 5's real results were not found "
          "in this environment when Notebook 39 ran.)")
print("\n✅ Section 8 complete.")


# =============================================================================
# SECTION 9: WORD REPORT -- Financial_Impact_Report.docx
# =============================================================================
_section("SECTION 9: Word Report -- Financial_Impact_Report.docx")


def _add_heading(doc, text, level=1):
    return doc.add_heading(text, level=level)


def _add_kv_table(doc, data: dict):
    table = doc.add_table(rows=0, cols=2)
    table.style = "Light Grid Accent 1"
    for k, v in data.items():
        row = table.add_row().cells
        row[0].text = str(k).replace("_", " ").title()
        row[1].text = "" if v is None else str(v)
    return table


def _add_table_from_df(doc, df, max_rows=30):
    table = doc.add_table(rows=1, cols=len(df.columns))
    table.style = "Light Grid Accent 1"
    hdr = table.rows[0].cells
    for i, col in enumerate(df.columns):
        hdr[i].text = str(col).replace("_", " ").title()
    for _, row in df.head(max_rows).iterrows():
        cells_ = table.add_row().cells
        for i, col in enumerate(df.columns):
            cells_[i].text = "" if pd.isna(row[col]) else str(row[col])
    return table


doc = Document()
doc.add_heading("AMEX Enterprise Credit Risk Platform", level=0)
doc.add_paragraph("Phase 3, Problem 6: Dynamic / Behavioral Credit Scoring -- Financial Impact Report")
doc.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

_add_heading(doc, "1. Executive Summary", level=1)
doc.add_paragraph(
    f"The trailing-window recency model (W={WINNING_W}, {CHAMPION_NAME}) validated in Notebooks 38-40 "
    f"re-scores an EXISTING book using only a customer's most recent {WINNING_W} statements, retaining "
    f"{WINNING_W_RESULT['auc_retention_pct_of_full_history']:.1f}% of the full-history model's real AUC. "
    f"This model is currently {'RECOMMENDED' if RECOMMENDED_FOR_PRODUCTION else 'NOT RECOMMENDED'} for "
    f"production. At the F1-optimal operating threshold on the real {N_HOLDOUT:,}-customer holdout, this "
    f"model correctly flags {TRUE_POSITIVES_FLAGGED:,} of {N_HOLDOUT_DEFAULTERS:,} real defaulters "
    f"({RECENCY_CAPTURE_RATE:.1%} capture) -- an exact, measured count, not an estimate -- alongside "
    f"{FALSE_POSITIVES_FLAGGED:,} false positives. At an ASSUMPTION {INTERVENTION_SUCCESS_RATE:.0%} "
    f"intervention success rate, net of an ASSUMPTION ${FALSE_POSITIVE_REVIEW_COST_USD} per-false-positive "
    f"review cost, this is estimated to net ${NET_BENEFIT_PER_CYCLE_USD:,.0f} of benefit per monthly "
    f"scoring cycle, for an estimated {PAYBACK_DISPLAY} payback on a ${IMPLEMENTATION_COST_USD:,.0f} "
    f"implementation investment."
)

_add_heading(doc, "2. Real Recency-Detection Value: Population Flagged", level=1)
doc.add_paragraph(
    "Every count in this section comes directly from Notebook 40's reproduced confusion matrix on the "
    "real holdout population -- exact, not derived or estimated."
)
_add_kv_table(doc, {
    "winning_trailing_window_w": WINNING_W,
    "f1_optimal_threshold": f"{METRICS_AT_F1_OPTIMAL['threshold']:.4f}",
    "real_holdout_population": f"{N_HOLDOUT:,}",
    "real_holdout_defaulters": f"{N_HOLDOUT_DEFAULTERS:,}",
    "true_positives_flagged": f"{TRUE_POSITIVES_FLAGGED:,}",
    "false_positives_flagged": f"{FALSE_POSITIVES_FLAGGED:,}",
    "real_defaulter_capture_rate": f"{RECENCY_CAPTURE_RATE:.1%}",
})

_add_heading(doc, "3. Loss-Prevention Opportunity, Net of False-Positive Review Cost", level=1)
_add_kv_table(doc, {
    "true_positives_flagged": TRUE_POSITIVES_FLAGGED,
    "intervention_success_rate_assumption": f"{INTERVENTION_SUCCESS_RATE:.0%}",
    "preventable_defaults": PREVENTABLE_DEFAULTS,
    "gross_loss_prevented_per_cycle_usd": f"${GROSS_LOSS_PREVENTED_USD:,.0f}",
    "false_positives_flagged": FALSE_POSITIVES_FLAGGED,
    "false_positive_review_cost_per_account_assumption": f"${FALSE_POSITIVE_REVIEW_COST_USD}",
    "total_false_positive_review_cost_usd": f"${FALSE_POSITIVE_COST_USD:,.0f}",
    "net_benefit_per_cycle_usd": f"${NET_BENEFIT_PER_CYCLE_USD:,.0f}",
})

_add_heading(doc, "4. ROI, Investment & Payback", level=1)
_add_kv_table(doc, {"amount_invested_usd": f"${IMPLEMENTATION_COST_USD:,.0f}",
                     "annual_net_benefit_usd": f"${ANNUAL_BENEFIT_USD:,.0f}",
                     "roi_year_1_pct": ROI_DISPLAY, "payback_period_months": PAYBACK_DISPLAY})

_add_heading(doc, "5. SMART Suggestions by Organizational Level", level=1)
_add_table_from_df(doc, smart_df)

_add_heading(doc, "6. Assumptions & Sources", level=1)
_assump_df = pd.DataFrame([{"assumption": k, "value": v["value"], "source": v["source"]}
                            for k, v in FINANCIAL_ASSUMPTIONS.items()])
_add_table_from_df(doc, _assump_df)

_add_heading(doc, "7. Charts", level=1)
doc.add_picture(str(chart1_path), width=Inches(6.0))
_p1 = doc.add_paragraph("Population flagged at the F1-optimal threshold (real, exact counts)")
_p1.alignment = WD_ALIGN_PARAGRAPH.CENTER
if chart2_path is not None:
    doc.add_picture(str(chart2_path), width=Inches(6.0))
    _p2 = doc.add_paragraph("Recent (Problem 6) vs. early (Problem 5) behavioral signal, same window length")
    _p2.alignment = WD_ALIGN_PARAGRAPH.CENTER

report_path = PILLAR_DIRS["p6_reporting_packaging"] / "Financial_Impact_Report.docx"
doc.save(str(report_path))
print(f"✅ Saved -> {report_path.name}")
print("\n✅ Section 9 complete.")


# =============================================================================
# SECTION 10: EXCEL WORKBOOK -- COLORFUL, TABLE + AUTOFILTER + CONDITIONAL FORMATTING + CHART
# =============================================================================
_section("SECTION 10: Excel Workbook -- Colorful, Table + AutoFilter + Conditional Formatting + Chart")

INK = "0B1F3A"
ACCENT = "C41E3A"
GOLD = "C9A227"
LIGHT = "F2F4F8"
WHITE = "FFFFFF"
USD_FMT = '$#,##0;($#,##0);-'

_assump_rows = {k: 2 + i for i, k in enumerate(FINANCIAL_ASSUMPTIONS.keys())}

wb = openpyxl.Workbook()

# --- Sheet 1: Assumptions (built first -- every formula below references these cells) ---
ws_assump = wb.active
ws_assump.title = "Assumptions"
ws_assump.append(["Assumption", "Value", "Source / Rationale"])
for _k, _v in FINANCIAL_ASSUMPTIONS.items():
    ws_assump.append([_k.replace("_", " ").title(), _v["value"], _v["source"]])
for _r in range(2, ws_assump.max_row + 1):
    ws_assump[f"B{_r}"].fill = PatternFill("solid", fgColor="FFFF00")
    ws_assump[f"B{_r}"].font = Font(name="Calibri", color="0000FF")
    ws_assump[f"C{_r}"].alignment = Alignment(wrap_text=True, vertical="top")
ws_assump[f"B{_assump_rows['lgd_assumption']}"].number_format = "0.0%"
ws_assump[f"B{_assump_rows['behavioral_intervention_success_rate']}"].number_format = "0.0%"
ws_assump[f"B{_assump_rows['ead_per_account_usd']}"].number_format = USD_FMT
ws_assump[f"B{_assump_rows['false_positive_review_cost_usd']}"].number_format = USD_FMT
ws_assump[f"B{_assump_rows['implementation_cost_usd']}"].number_format = USD_FMT
ws_assump.column_dimensions["A"].width = 36
ws_assump.column_dimensions["B"].width = 14
ws_assump.column_dimensions["C"].width = 90
_tbl_assump = Table(displayName="Assumptions", ref=f"A1:C{ws_assump.max_row}")
_tbl_assump.tableStyleInfo = TableStyleInfo(name="TableStyleMedium4", showRowStripes=True)
ws_assump.add_table(_tbl_assump)

_ead_ref = f"Assumptions!$B${_assump_rows['ead_per_account_usd']}"
_lgd_ref = f"Assumptions!$B${_assump_rows['lgd_assumption']}"
_intervention_rate_ref = f"Assumptions!$B${_assump_rows['behavioral_intervention_success_rate']}"
_fp_cost_ref = f"Assumptions!$B${_assump_rows['false_positive_review_cost_usd']}"
_cost_ref = f"Assumptions!$B${_assump_rows['implementation_cost_usd']}"
_cycles_ref = f"Assumptions!$B${_assump_rows['annual_application_cycles']}"

# --- Sheet 2: Recency Impact -- key figures as REAL FORMULAS referencing Assumptions,
#     so editing an assumption recalculates every dollar figure. ---
ws_impact = wb.create_sheet("Recency Impact")
ws_impact.append(["Metric", "Value"])
_impact_rows_static = [
    ("Winning Trailing Window W (Notebook 40)", WINNING_W),
    ("Real Holdout Population", N_HOLDOUT),
    ("Real Holdout Defaulters (exact)", N_HOLDOUT_DEFAULTERS),
    ("True Positives Flagged (exact)", TRUE_POSITIVES_FLAGGED),
    ("False Positives Flagged (exact)", FALSE_POSITIVES_FLAGGED),
    ("Real Defaulter Capture Rate", RECENCY_CAPTURE_RATE),
]
for _label, _val in _impact_rows_static:
    ws_impact.append([_label, _val])
_preventable_row = ws_impact.max_row + 1
ws_impact.append(["Preventable Defaults", f"=ROUND(B5*{_intervention_rate_ref},0)"])
_gross_loss_row = ws_impact.max_row + 1
ws_impact.append(["Gross Loss Prevented / Cycle (USD)", f"=B{_preventable_row}*{_ead_ref}*{_lgd_ref}"])
_fp_cost_row = ws_impact.max_row + 1
ws_impact.append(["False-Positive Review Cost / Cycle (USD)", f"=B6*{_fp_cost_ref}"])
_net_benefit_row = ws_impact.max_row + 1
ws_impact.append(["Net Benefit / Cycle (USD)", f"=B{_gross_loss_row}-B{_fp_cost_row}"])
_annual_benefit_row = ws_impact.max_row + 1
ws_impact.append(["Annual Net Benefit (USD)", f"=B{_net_benefit_row}*{_cycles_ref}"])
ws_impact["B7"].number_format = "0.0%"
ws_impact[f"B{_gross_loss_row}"].number_format = USD_FMT
ws_impact[f"B{_fp_cost_row}"].number_format = USD_FMT
ws_impact[f"B{_net_benefit_row}"].number_format = USD_FMT
ws_impact[f"B{_annual_benefit_row}"].number_format = USD_FMT
ws_impact.column_dimensions["A"].width = 42
ws_impact.column_dimensions["B"].width = 20
_tbl_impact = Table(displayName="RecencyImpact", ref=f"A1:B{ws_impact.max_row}")
_tbl_impact.tableStyleInfo = TableStyleInfo(name="TableStyleMedium2", showRowStripes=True)
ws_impact.add_table(_tbl_impact)

_chart = BarChart()
_chart.title = "Flagged Population: True Positives vs. False Positives"
_chart.y_axis.title = "Customers"
_data = Reference(ws_impact, min_col=2, min_row=1, max_row=6)
_cats = Reference(ws_impact, min_col=1, min_row=2, max_row=6)
_chart.add_data(_data, titles_from_data=True)
_chart.set_categories(_cats)
_chart.width, _chart.height = 20, 10
ws_impact.add_chart(_chart, "D2")

# --- Sheet 3: SMART Suggestions (real Excel Table -> native AutoFilter dropdowns) ---
ws_smart = wb.create_sheet("SMART Suggestions")
ws_smart.append(["Org Level", "Suggestion"])
for _row_data in SMART_SUGGESTIONS:
    ws_smart.append([_row_data["org_level"], _row_data["suggestion"]])
_last_row_smart = ws_smart.max_row
_tbl_smart = Table(displayName="SmartSuggestions", ref=f"A1:B{_last_row_smart}")
_tbl_smart.tableStyleInfo = TableStyleInfo(name="TableStyleMedium7", showRowStripes=True)
ws_smart.add_table(_tbl_smart)
ws_smart.column_dimensions["A"].width = 40
ws_smart.column_dimensions["B"].width = 100
for _r in range(2, _last_row_smart + 1):
    ws_smart[f"B{_r}"].alignment = Alignment(wrap_text=True, vertical="top")

# --- Sheet 4: Executive Summary (KPI cards), inserted first, populated last ---
ws_exec = wb.create_sheet("Executive Summary", 0)
wb.active = 0
ws_exec.sheet_view.showGridLines = False
ws_exec["B2"] = "AMEX RiskIQ -- Problem 6: Dynamic / Behavioral Credit Scoring"
ws_exec["B2"].font = Font(name="Calibri", size=16, bold=True, color=WHITE)
ws_exec["B2"].fill = PatternFill("solid", fgColor=INK)
ws_exec.merge_cells("B2:F2")
ws_exec["B3"] = "Financial Impact Summary"
ws_exec["B3"].font = Font(name="Calibri", size=11, italic=True, color=INK)
ws_exec.merge_cells("B3:F3")

_kpi_rows = [
    ("Trailing Window (W)", f"W={WINNING_W}  (reported, see Recency Impact sheet)", False, LIGHT),
    ("Defaulters Captured (Exact)", "='Recency Impact'!B4", True, LIGHT),
    ("Net Benefit / Cycle", f"='Recency Impact'!B{_net_benefit_row}", True, GOLD),
    ("Amount Invested", f"=\"$\"&TEXT({_cost_ref},\"#,##0\")", True, LIGHT),
    ("Estimated Year-1 ROI", f"{ROI_DISPLAY}  (reported, see Section 6)", False, ACCENT),
    ("Estimated Payback", f"{PAYBACK_DISPLAY}  (reported, see Section 6)", False, ACCENT),
    ("Deployment Status", f"{'RECOMMENDED' if RECOMMENDED_FOR_PRODUCTION else 'NOT RECOMMENDED'} for production", False, ACCENT if not RECOMMENDED_FOR_PRODUCTION else "63BE7B"),
]
_row = 5
for _label, _value, _is_formula, _fill in _kpi_rows:
    ws_exec.cell(row=_row, column=2, value=_label).font = Font(name="Calibri", size=11, color=INK)
    _cell = ws_exec.cell(row=_row, column=4, value=_value)
    _cell.font = Font(name="Calibri", size=13, bold=True, color=(WHITE if _fill in (GOLD, ACCENT) else INK))
    _cell.fill = PatternFill("solid", fgColor=_fill)
    _cell.alignment = Alignment(horizontal="center", wrap_text=not _is_formula)
    if _is_formula and _label in ("Defaulters Captured (Exact)",):
        _cell.number_format = "#,##0"
    elif _is_formula and "Benefit" in _label:
        _cell.number_format = USD_FMT
    ws_exec.merge_cells(start_row=_row, start_column=4, end_row=_row, end_column=5)
    _row += 1
ws_exec["B14"] = "Rows 6-8 recalculate live from the Assumptions and Recency Impact sheets."
ws_exec["B14"].font = Font(name="Calibri", size=9, italic=True, color="8A93A6")
ws_exec.merge_cells("B14:F14")
for _col, _w in zip("BCDEF", [32, 3, 22, 22, 3]):
    ws_exec.column_dimensions[_col].width = _w

for _ws in (ws_impact, ws_smart, ws_assump):
    for _cell in _ws[1]:
        _cell.font = Font(name="Calibri", bold=True, color=WHITE)
        _cell.fill = PatternFill("solid", fgColor=INK)

workbook_path = PILLAR_DIRS["p6_reporting_packaging"] / "AMEX_Problem6_Financial_Impact_Workbook.xlsx"
wb.save(str(workbook_path))
print(f"✅ Saved -> {workbook_path.name}")
print("\n✅ Section 10 complete.")


# =============================================================================
# SECTION 11: INTERACTIVE HTML DASHBOARD
# =============================================================================
_section("SECTION 11: Interactive HTML Dashboard")

_smart_json = json.dumps(SMART_SUGGESTIONS)
_org_levels = sorted({r["org_level"] for r in SMART_SUGGESTIONS})

_html = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Problem 6 -- Financial Impact Dashboard</title>
<script src="https://cdn.jsdelivr.net/npm/chart.js@4"></script>
<style>
  :root { --ink:#0B1F3A; --accent:#C41E3A; --gold:#C9A227; --muted:#8A93A6; --bg:#F2F4F8; --card:#FFFFFF; }
  body { font-family: Calibri, Arial, sans-serif; background: var(--bg); color: var(--ink); margin: 0; padding: 24px; }
  h1 { font-size: 22px; margin-bottom: 4px; }
  .sub { color: var(--muted); margin-bottom: 20px; }
  .kpi-row { display: flex; flex-wrap: wrap; gap: 14px; margin-bottom: 24px; }
  .kpi { background: var(--card); border-radius: 10px; padding: 16px 20px; box-shadow: 0 1px 3px rgba(0,0,0,.12); min-width: 190px; flex: 1; }
  .kpi .label { font-size: 12px; color: var(--muted); text-transform: uppercase; }
  .kpi .value { font-size: 22px; font-weight: 700; margin-top: 4px; }
  .panel { background: var(--card); border-radius: 10px; padding: 18px; margin-bottom: 20px; box-shadow: 0 1px 3px rgba(0,0,0,.12); }
  table { width: 100%; border-collapse: collapse; font-size: 13px; }
  th, td { text-align: left; padding: 8px 10px; border-bottom: 1px solid #E4E7EE; }
  th { background: var(--ink); color: #fff; }
  select { padding: 6px 10px; border-radius: 6px; border: 1px solid var(--muted); font-size: 13px; margin-bottom: 12px; }
  canvas { max-height: 360px; }
  .badge { display: inline-block; padding: 3px 10px; border-radius: 12px; font-size: 12px; font-weight: 700; }
</style>
</head>
<body>
<h1>AMEX RiskIQ -- Problem 6: Dynamic / Behavioral Credit Scoring</h1>
<div class="sub">Financial Impact Dashboard -- real Notebook 38-40 results, ASSUMPTION values clearly marked</div>

<div class="kpi-row">
  <div class="kpi"><div class="label">Trailing Window</div><div class="value">W=__WINNING_W__</div></div>
  <div class="kpi"><div class="label">Defaulters Captured (Exact)</div><div class="value">__TP_FLAGGED__</div></div>
  <div class="kpi"><div class="label">Net Benefit / Cycle</div><div class="value">__NET_BENEFIT__</div></div>
  <div class="kpi"><div class="label">Est. Year-1 ROI</div><div class="value">__ROI__</div></div>
  <div class="kpi"><div class="label">Est. Payback</div><div class="value">__PAYBACK__</div></div>
  <div class="kpi"><div class="label">Deployment Status</div><div class="value"><span class="badge" style="background:__STATUS_COLOR__;color:#fff;">__STATUS__</span></div></div>
</div>

<div class="panel">
  <canvas id="captureChart"></canvas>
</div>

<div class="panel">
  <label for="orgFilter"><b>SMART Suggestions -- filter by organizational level</b></label><br/>
  <select id="orgFilter"></select>
  <table id="smartTable"><thead><tr><th>Org Level</th><th>Suggestion</th></tr></thead><tbody></tbody></table>
</div>

<script>
const smartData = __SMART_JSON__;
const orgLevels = __ORG_LEVELS__;

const ctx = document.getElementById("captureChart").getContext("2d");
new Chart(ctx, {
  type: "bar",
  data: {
    labels: ["True Positives (defaulters flagged)", "False Positives (review cost)"],
    datasets: [{ data: [__TP__, __FP__], backgroundColor: ["#C41E3A", "#8A93A6"] }],
  },
  options: { responsive: true, plugins: { legend: { display: false } },
             scales: { y: { ticks: { callback: v => v.toLocaleString() } } } },
});

function renderSmart(filterLevel) {
  const tbody = document.querySelector("#smartTable tbody");
  tbody.innerHTML = "";
  smartData.filter(r => filterLevel === "All" || r.org_level === filterLevel).forEach(r => {
    const tr = document.createElement("tr");
    tr.innerHTML = `<td>${r.org_level}</td><td>${r.suggestion}</td>`;
    tbody.appendChild(tr);
  });
}

const orgSelect = document.getElementById("orgFilter");
["All", ...orgLevels].forEach(level => {
  const opt = document.createElement("option");
  opt.value = level; opt.textContent = level;
  orgSelect.appendChild(opt);
});
orgSelect.onchange = () => renderSmart(orgSelect.value);
renderSmart("All");
</script>
</body>
</html>
"""
_html = (_html
         .replace("__WINNING_W__", str(WINNING_W))
         .replace("__TP_FLAGGED__", f"{TRUE_POSITIVES_FLAGGED:,}")
         .replace("__NET_BENEFIT__", f"${NET_BENEFIT_PER_CYCLE_USD:,.0f}")
         .replace("__ROI__", ROI_DISPLAY)
         .replace("__PAYBACK__", PAYBACK_DISPLAY)
         .replace("__STATUS__", "RECOMMENDED" if RECOMMENDED_FOR_PRODUCTION else "NOT RECOMMENDED")
         .replace("__STATUS_COLOR__", "#16a34a" if RECOMMENDED_FOR_PRODUCTION else "#dc2626")
         .replace("__TP__", str(TRUE_POSITIVES_FLAGGED))
         .replace("__FP__", str(FALSE_POSITIVES_FLAGGED))
         .replace("__SMART_JSON__", _smart_json)
         .replace("__ORG_LEVELS__", json.dumps(_org_levels)))

dashboard_path = PILLAR_DIRS["p6_reporting_packaging"] / "financial_impact_dashboard.html"
with open(dashboard_path, "w", encoding="utf-8") as f:
    f.write(_html)
print(f"✅ Saved -> {dashboard_path.name}")
print("\n✅ Section 11 complete.")


# =============================================================================
# SECTION 12: VERIFICATION
# =============================================================================
_section("SECTION 12: Verification")

_checks_passed = True


def _check(label, condition, detail=""):
    global _checks_passed
    if condition:
        print(f"✅ {label}")
    else:
        _checks_passed = False
        print(f"❌ {label}  {detail}")


_check("True positives + false negatives equals the real holdout defaulter count",
       TRUE_POSITIVES_FLAGGED + _cm_f1["fn"] == N_HOLDOUT_DEFAULTERS)
_check("Preventable defaults does not exceed true positives flagged",
       PREVENTABLE_DEFAULTS <= TRUE_POSITIVES_FLAGGED)
_check("Net benefit per cycle equals gross loss prevented minus false-positive review cost",
       abs(NET_BENEFIT_PER_CYCLE_USD - (GROSS_LOSS_PREVENTED_USD - FALSE_POSITIVE_COST_USD)) < 1e-6)
_check("ROI is a finite number (implementation cost is a fixed, non-zero assumption)",
       ROI_PCT is not None)
_check("Payback is a positive finite number when there is measurable annual net benefit, "
       "and explicitly undefined (None) otherwise -- never a crash or a fabricated value",
       (PAYBACK_MONTHS is not None and PAYBACK_MONTHS > 0) if ANNUAL_BENEFIT_USD > 0
       else PAYBACK_MONTHS is None)
_check("EAD/LGD were inherited from Notebook 08, not re-guessed",
       EAD_PER_ACCOUNT_USD == NB08_SUMMARY["ead_per_account_usd_assumption"]
       and LGD_ASSUMPTION == NB08_SUMMARY["lgd_assumption"])
_check("Confusion-matrix counts were reused verbatim from Notebook 40 (not re-derived)",
       TRUE_POSITIVES_FLAGGED == NB40_SUMMARY["metrics_at_f1_optimal_threshold"]["confusion_matrix"]["tp"])

_expected_files = [assumptions_path, smart_path, chart1_path, report_path, workbook_path, dashboard_path]
if chart2_path is not None:
    _expected_files.append(chart2_path)
for fp in _expected_files:
    _check(f"{fp.name} exists and is non-empty", fp.exists() and fp.stat().st_size > 0)

if not _checks_passed:
    raise RuntimeError("One or more Notebook 41 verification checks failed. See ❌ line above.")
print("\nAll Notebook 41 checks passed.")
print("\n✅ Section 12 complete.")


# =============================================================================
# SECTION 13: WRITE NOTEBOOK 41 SUMMARY -- PROBLEM 6 COMPLETE
# =============================================================================
_section("SECTION 13: Write Notebook 41 Summary -- Problem 6 Complete")

notebook_41_summary = {
    "notebook": "41_dynamic_behavioral_scoring_financial_impact_reporting_packaging",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem_number": 6, "problem_name": "Dynamic / Behavioral Credit Scoring",
    "phase": "Phase 3 -- Behavioral Intelligence", "problem_6_complete": True,
    "winning_w": WINNING_W, "meets_kpi_target": MEETS_KPI, "recommended_for_production": RECOMMENDED_FOR_PRODUCTION,
    "true_positives_flagged": TRUE_POSITIVES_FLAGGED, "false_positives_flagged": FALSE_POSITIVES_FLAGGED,
    "real_defaulter_capture_rate": round(RECENCY_CAPTURE_RATE, 4),
    "net_benefit_per_cycle_usd": round(NET_BENEFIT_PER_CYCLE_USD, 2),
    "roi_year_1_pct": ROI_PCT_JSON, "payback_period_months": PAYBACK_MONTHS_JSON,
    "output_files": {p.name: str(p) for p in _expected_files},
}
nb41_summary_path = ARTIFACTS_DIR / "notebook_41_summary.json"
with open(nb41_summary_path, "w", encoding="utf-8") as f:
    json.dump(notebook_41_summary, f, indent=2)
print(f"✅ Saved -> {nb41_summary_path.name}")
print("\n✅ Section 13 complete.")


# =============================================================================
# SECTION 14: COMPLETION SUMMARY -- PROBLEM 6 COMPLETE
# =============================================================================
_section("SECTION 14: Notebook 41 Complete -- Problem 6 Complete")

print("NOTEBOOK 41: FINANCIAL-IMPACT REPORTING & PACKAGING -- COMPLETE")
print("PROBLEM 6 (DYNAMIC / BEHAVIORAL CREDIT SCORING) -- ALL 4 NOTEBOOKS COMPLETE")
print(f"  Winning window / meets KPI / recommended : W={WINNING_W} / {MEETS_KPI} / {RECOMMENDED_FOR_PRODUCTION}")
print(f"  Real defaulters captured (exact)            : {TRUE_POSITIVES_FLAGGED:,} of {N_HOLDOUT_DEFAULTERS:,} "
      f"({RECENCY_CAPTURE_RATE:.1%})")
print(f"  Net benefit per cycle                       : ${NET_BENEFIT_PER_CYCLE_USD:,.0f}")
print(f"  Estimated Year-1 ROI / payback               : {ROI_DISPLAY} / {PAYBACK_DISPLAY}")
print(f"  Files produced                              : {len(_expected_files) + 1}")
for _p in _expected_files + [nb41_summary_path]:
    print(f"    - {_p.name}")
print("\n  PROBLEM 6 (Dynamic/Behavioral Credit Scoring) is now complete. Next: Problem 7 (Early Warning "
      "System), Notebooks 42-45 -- depends on Problem 6's real results (rolling z-score trend-deviation "
      "detection).")
print("\n✅ Ready to proceed.")
